# Clasificación de Variedades de Frijol Seco mediante Aprendizaje Automático (Dry Bean Dataset)

## A. Presentación y Descripción del Problema

* **Dataset elegido:** Dry Bean Dataset (Clasificación morfológica de frijol seco).
* **Fuente:** OpenML (`fetch_openml("DryBeanDataset")`) / UCI Machine Learning Repository.
* **Tipo de problema:** Clasificación multiclase supervisada.
* **Objetivo:** Clasificar muestras de granos de frijol seco en **7 variedades** (**Seker**, **Barbunya**, **Bombay**, **Cali**, **Dermosan**, **Horoz**, **Sira**) a partir de características geométricas y de forma extraídas mediante visión por computadora.

## B. Carga y Análisis Exploratorio de Datos (EDA)

### B.1 Carga del dataset y normalización de nombres

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

# Carga automática del dataset Dry Bean desde OpenML (con respaldo si el API responde distinto)
try:
    datos = fetch_openml("DryBeanDataset", as_frame=True, parser="auto")
    df = datos.frame.copy()
except Exception:
    url_csv = "https://archive.ics.uci.edu/ml/machine-learning-databases/00602/DryBeanDataset/Dry_Bean_Dataset.xlsx"
    df = pd.read_excel(url_csv)

# Estandarizamos el nombre de la variable objetivo a 'class'
objetivo_candidatos = ["Class", "class", "target"]
col_objetivo = next((c for c in objetivo_candidatos if c in df.columns), df.columns[-1])
if col_objetivo != "class":
    df = df.rename(columns={col_objetivo: "class"})

df["class"] = df["class"].astype(str).str.strip().str.capitalize()

print("Dataset cargado:", df.shape)

### B.2 Vista general, tipos de datos, nulos y duplicados

In [ ]:
pd.set_option("display.max_columns", None)
display(df.head())

print("Filas y columnas:", df.shape)
df.info()
print("\nTipos de datos:\n", df.dtypes.value_counts())

# Valores nulos y duplicados
nulos = df.isna().sum()
print("\nColumnas con valores nulos:\n", nulos[nulos > 0])
print("Total de valores nulos:", nulos.sum())
print("Filas repetidas:", df.duplicated().sum())

### B.3 Distribución de la variable objetivo (`class`)

In [ ]:
orden_clases = ["Seker", "Barbunya", "Bombay", "Cali", "Dermosan", "Horoz", "Sira"]
presentes = [c for c in orden_clases if c in df["class"].unique()]
resto = [c for c in sorted(df["class"].dropna().unique()) if c not in presentes]
orden_final = presentes + resto

conteo = df["class"].value_counts().reindex(orden_final).dropna().astype(int)
porcentaje = (conteo / len(df)) * 100

resumen_objetivo = pd.DataFrame({
    "conteo": conteo,
    "porcentaje (%)": porcentaje.round(2)
})
display(resumen_objetivo)

fig, ax = plt.subplots(figsize=(9, 5))
colores = ["#2E86AB", "#C73E1D", "#F18F01", "#A23B72", "#4C9F70", "#5C4D7D", "#E07A5F"]
barras = ax.bar(conteo.index, conteo.values, color=colores[:len(conteo)])

for barra, n, p in zip(barras, conteo.values, porcentaje.values):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height(),
        f"{n:,}\n({p:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_title("Distribución de la variable objetivo (Variedad de frijol seco)")
ax.set_xlabel("Variedad (class)")
ax.set_ylabel("Número de muestras")
ax.set_ylim(0, conteo.max() * 1.18)
plt.tight_layout()
plt.show()

### B.4 Variables categóricas y binarias

In [ ]:
categoricas = [
    col for col in df.columns
    if col != "class" and (df[col].dtype == "object" or str(df[col].dtype) == "category")
]
binarias = [
    col for col in df.columns
    if col != "class" and col not in categoricas and df[col].dropna().nunique() <= 2
]

print("Variables categóricas:", len(categoricas), categoricas)
for c in categoricas:
    print(f"\n--- {c} (top 10) ---")
    print(df[c].value_counts(dropna=False).head(10))

print("\nVariables binarias:", len(binarias))
for b in binarias:
    vals = df[b].value_counts().to_dict()
    print(f"  {b}: {vals}")

### B.5 Variables numéricas (estadísticas descriptivas y asimetría)

In [ ]:
excluir = set(categoricas + binarias + ["class"])
numericas = [col for col in df.columns if col not in excluir and pd.api.types.is_numeric_dtype(df[col])]

print("Variables numéricas:", len(numericas))
display(df[numericas].describe(percentiles=[0.25, 0.5, 0.75, 0.99]).round(2))

asimetria = df[numericas].skew().sort_values(ascending=False)
print("\nTop 10 variables numéricas con mayor asimetría:")
print(asimetria.head(10).round(2))

### B.6 Gráficos exploratorios clave

In [ ]:
# B.6.1 Distribución de una variable numérica clave (Área del grano)
var_area = "Area" if "Area" in df.columns else numericas[0]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[var_area].dropna(), bins=40, color="#4C72B0", edgecolor="white")
ax.set_title(f"Distribución de la variable {var_area}")
ax.set_xlabel(var_area)
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

# B.6.2 Relación entre variedad de frijol y variable morfológica clave (Boxplot)
var_forma = "MajorAxisLength" if "MajorAxisLength" in df.columns else numericas[min(1, len(numericas) - 1)]
datos_box = [df.loc[df["class"] == k, var_forma].dropna().values for k in conteo.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(datos_box, labels=conteo.index, patch_artist=True)
ax.set_title(f"Distribución de {var_forma} según variedad de frijol (class)")
ax.set_xlabel("Variedad (class)")
ax.set_ylabel(var_forma)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### B.7 Matriz de correlación en variables numéricas

In [ ]:
if len(numericas) > 1:
    corr = df[numericas].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(numericas)))
    ax.set_yticks(range(len(numericas)))
    ax.set_xticklabels(numericas, rotation=90, fontsize=8)
    ax.set_yticklabels(numericas, fontsize=8)
    ax.set_title("Matriz de correlación (variables numéricas)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## C. Preprocesamiento de Datos

### C.1 Limpieza, división Train/Test y codificación de la variable objetivo

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df_limpio = df.dropna(subset=["class"]).drop_duplicates().copy()

X = df_limpio.drop(columns=["class"])
y = df_limpio["class"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Clases codificadas:", dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")

### C.2 Pipeline de transformación (Escalamiento + One-Hot Encoding)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

cols_cat = [c for c in categoricas if c in X_train.columns]
cols_num = [c for c in X_train.columns if c not in cols_cat]

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocesador = ColumnTransformer(
    transformers=[
        ("num", num_transformer, cols_num),
        ("cat", cat_transformer, cols_cat)
    ]
)

print("Preprocesador configurado correctamente.")

## D. Modelado y Evaluación Comparativa

### D.1 Entrenamiento y comparación de 3 modelos de clasificación

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=150, random_state=42)
}

resultados = []
pipelines_entrenados = {}

for nombre, modelo in modelos.items():
    inicio = time.time()
    
    clf = Pipeline(steps=[
        ("preprocessor", preprocesador),
        ("classifier", modelo)
    ])
    
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    duracion = time.time() - inicio
    pipelines_entrenados[nombre] = clf
    
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (Macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-Score (Macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-Score (Weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "Tiempo (s)": round(duracion, 2)
    })

df_resultados = pd.DataFrame(resultados).sort_values(by="F1-Score (Macro)", ascending=False)
display(df_resultados.round(4))

### D.2 Gráfico comparativo de desempeño entre modelos

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df_resultados))
ancho = 0.28

ax.bar(x - ancho, df_resultados["Accuracy"], width=ancho, label="Accuracy", color="#2E86AB")
ax.bar(x, df_resultados["F1-Score (Weighted)"], width=ancho, label="F1 (Weighted)", color="#4C9F70")
ax.bar(x + ancho, df_resultados["F1-Score (Macro)"], width=ancho, label="F1 (Macro)", color="#F18F01")

ax.set_xticks(x)
ax.set_xticklabels(df_resultados["Modelo"])
ax.set_ylim(0, 1.1)
ax.set_ylabel("Puntuación")
ax.set_title("Comparación de métricas por modelo en Dry Bean Dataset")
ax.legend()
plt.tight_layout()
plt.show()

### D.3 Evaluación detallada del mejor modelo (Matriz de Confusión y Reporte)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

mejor_nombre = df_resultados.iloc[0]["Modelo"]
mejor_modelo = pipelines_entrenados[mejor_nombre]
y_pred_best = mejor_modelo.predict(X_test)

print(f"=== Reporte de Clasificación: {mejor_nombre} ===\n")
print(classification_report(y_test, y_pred_best, target_names=le.classes_, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_best, normalize="true")
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap="Blues", values_format=".2f", ax=ax)
ax.set_title(f"Matriz de Confusión Normalizada - {mejor_nombre}")
plt.tight_layout()
plt.show()

### D.4 Importancia de características (Feature Importance)

In [ ]:
rf_pipe = pipelines_entrenados["Random Forest"]
nombres_features = rf_pipe.named_steps["preprocessor"].get_feature_names_out()
importancias = rf_pipe.named_steps["classifier"].feature_importances_

df_imp = pd.DataFrame({
    "Feature": [f.replace("num__", "").replace("cat__", "") for f in nombres_features],
    "Importancia": importancias
}).sort_values(by="Importancia", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_imp["Feature"][::-1], df_imp["Importancia"][::-1], color="#2E86AB")
ax.set_title("Top 15 variables más importantes para clasificar variedades de frijol (Random Forest)")
ax.set_xlabel("Importancia relativa")
plt.tight_layout()
plt.show()

## E. Conclusiones

1. **Separabilidad morfológica entre variedades:** En el **Dry Bean Dataset**, las dimensiones geométricas (`Area`, `Perimeter`, `MajorAxisLength`, `MinorAxisLength`) y los descriptores de forma (`ShapeFactor1`–`ShapeFactor4`, `Eccentricity`, `Extent`) permiten distinguir con alta precisión las 7 variedades de frijol seco, aunque existen similitudes naturales entre clases cercanas como `Sira` y `Dermosan`.
2. **Desempeño de modelos:** Los modelos no lineales y de ensamble (**Random Forest** e **HistGradientBoosting**) superan a la **Regresión Logística** al capturar fronteras de decisión complejas entre variables geométricas altamente correlacionadas.
3. **Importancia de métricas multiclase:** Revisar en conjunto el **F1-Score Macro**, el **F1-Score Weighted** y la matriz de confusión normalizada evita que una clase mayoritaria oculte confusiones puntuales en variedades con menor cantidad de muestras.

## F. Limitaciones y Mejoras Futuras

### F.1 Limitaciones identificadas
* **Alta colinealidad entre variables geométricas:** Atributos como `Area`, `Perimeter`, `ConvexArea`, `MajorAxisLength` y `MinorAxisLength` miden aspectos muy relacionados del tamaño del grano, lo que genera correlaciones elevadas y redundancia de información.
* **Sensibilidad a condiciones de captura y segmentación:** Las variables morfológicas dependen de la calibración de cámara, iluminación, escala espacial y umbral de segmentación del grano; cambios en el sistema de visión artificial en producción podrían desplazar las distribuciones de entrada (*data drift*).
* **Desbalanceo moderado entre variedades:** Clases como `Bombay` tienen considerablemente menos muestras que `Dermosan` o `Sira`, por lo que errores en clases minoritarias pesan poco en métricas globales ponderadas si no se inspecciona el desempeño por clase.

### F.2 Propuestas de mejora
* **Validación cruzada estratificada y ajuste de hiperparámetros:** Aplicar `StratifiedKFold` junto con `GridSearchCV` o `RandomizedSearchCV` para optimizar profundidad, número de árboles o tasa de aprendizaje y reducir el riesgo de sobreajuste en una única partición.
* **Ingeniería de características y selección de variables:** Evaluar relaciones adimensionales o de proporción (`MinorAxisLength / MajorAxisLength`, compacidad, redondez) y analizar si eliminar variables altamente redundantes mejora la estabilidad y velocidad de inferencia.
* **Monitoreo de drift en despliegue:** Registrar estadísticas de las variables de entrada en producción y establecer alertas ante desviaciones significativas respecto a los datos de entrenamiento.

### F.3 Verificación de estabilidad con validación cruzada estratificada

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

mejor_pipe = pipelines_entrenados[mejor_nombre]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_cv = cross_val_score(mejor_pipe, X, y_encoded, cv=cv, scoring="f1_macro", n_jobs=-1)

print(f"F1-Score Macro por fold ({mejor_nombre}):", np.round(scores_cv, 4))
print(f"Media CV F1-Macro: {scores_cv.mean():.4f} | Desviación estándar: {scores_cv.std():.4f}")

## G. Conclusiones Finales

* **Modelo recomendado:** Los modelos de ensamble basados en árboles (**Random Forest** e **HistGradientBoosting**) ofrecen el mejor equilibrio entre precisión multiclase, capacidad para modelar relaciones no lineales y robustez frente a variables geométricas correlacionadas en el **Dry Bean Dataset**.
* **Utilidad práctica:** Un clasificador morfológico automatizado permite acelerar el control de calidad y la clasificación varietal en procesos agroindustriales, reduciendo la subjetividad y el tiempo de la inspección visual manual.
* **Cierre del flujo experimental:** El pipeline reproducible —desde la exploración y limpieza hasta el preprocesamiento con `ColumnTransformer`, la comparación de modelos y la validación cruzada estratificada— sienta una base sólida para iteraciones futuras con ajuste fino de hiperparámetros y pruebas en entornos reales.